---
INFO
---
Name: Kiarash Akbari

Date: 1405-4-5

---
SECTION 1: Stats
---

<div dir="rtl" style="text-align: right;">
 
 **سوال ۱: ریسک L1 و L2**
 
الف) تو تابع L2 اگه مشتق بگیریم و مساوی صفر بذاریم y_hat همون میانگین درمیاد. تو L1 مشتق قدرمطلق میشه تابع علامت (sign) که صفر قرار دادنش یعنی احتمال چپ و راست برابره که این همون تعریف میانه‌ست.
ج) دیتای قیمت معمولا چوله به راسته و تراکنش پرت زیاد داره. تابع L2 چون توان دو داره خطای این پرت‌ها رو خیلی بزرگ میکنه ولی L1 (میانه) در برابر نویز و دیتای پرت مقاومه.

<div align="center">

</div>

</div>

In [2]:
# q1: l1 vs l2 risk
def mean(d): return sum(d) / len(d) if d else 0

def median(d):
    s = sorted(d)
    n = len(s)
    if n == 0: return 0
    return s[n//2] if n % 2 != 0 else (s[n//2 - 1] + s[n//2]) / 2.0

prices = [100, 200, 250, 50000] # 50000 is an outlier
m1, m2 = mean(prices), median(prices)
# median ignores the outlier better




<div dir="rtl" style="text-align: right;">
 


**سوال ۳: آزمون جایگشت تهران و شیراز**

الف) قابلیت معاوضه یعنی تحت فرض صفر فرقی نداره یوزر مال تهران باشه یا شیراز. پس اگه لیبل شهرها رو قاطی کنیم (جایگشت) نباید توزیع اختلاف میانگین‌ها عوض شه. این روش اصلا نیازی به فرض نرمال بودن نداره چون توزیع رو مستقیم از خود دیتا میسازیم. 

<div align="center">

</div>

</div>

In [8]:
# q3: permutation test
# simple random generator without random module
def lcg(seed=42):
    while True:
        seed = (1103515245 * seed + 12345) % (2**31)
        yield seed / (2**31)

def perm_test(tehran, shiraz, iters=1000):
    rng = lcg()
    obs_diff = abs(mean(tehran) - mean(shiraz))
    all_data = tehran + shiraz
    n = len(tehran)
    
    extreme_count = 0
    for _ in range(iters):
        # manual shuffle
        for i in range(len(all_data)-1, 0, -1):
            j = int(next(rng) * (i + 1))
            all_data[i], all_data[j] = all_data[j], all_data[i]
            
        p_tehran = all_data[:n]
        p_shiraz = all_data[n:]
        if abs(mean(p_tehran) - mean(p_shiraz)) >= obs_diff:
            extreme_count += 1
            
    return obs_diff, extreme_count / iters


<div dir="rtl" style="text-align: right;">

 ##  سوال ۴: بوت‌استرپ چولگی
 
الف) آماره‌هایی مثل چولگی فرمول گشتاوری پیچیده‌ای دارن و توزیعشون تو حجم نمونه کم اصلا شبیه نرمال نیست. روش کلاسیک اینجا جواب نمیده ولی بوت استرپ با بازنمونه‌گیری از خود دیتا توزیع رو راحت تقریب میزنه.

<div align="center">

</div>

</div>

In [4]:
# q4: bootstrap skewness
def skewness(d):
    m = mean(d)
    m2 = sum((x - m)**2 for x in d) / len(d)
    m3 = sum((x - m)**3 for x in d) / len(d)
    return m3 / (m2**1.5) if m2 else 0

def boot_skew(data, iters=1000):
    rng = lcg(123)
    n = len(data)
    skews = []
    
    for _ in range(iters):
        # sample with replacement
        samp = [data[int(next(rng) * n)] for _ in range(n)]
        skews.append(skewness(samp))
        
    skews.sort()
    return skews[int(iters * 0.025)], skews[int(iters * 0.975)]

In [13]:
#result q1
print("Q1 (L1 vs L2) -> Mean:", m1, "| Median:", m2)
print("نکته: میانه تحت تاثیر داده پرت (50000) قرار نگرفت.\n")

#result q3
tehran_sample = [2, 4, 5, 2, 6, 3, 5]
shiraz_sample = [1, 2, 1, 3, 2, 1, 2]
obs_diff, p_value = perm_test(tehran_sample, shiraz_sample, iters=1000)
print(f"Q3 (Permutation Test) -> Obs Diff: {obs_diff:.2f}, p-value: {p_value}\n")

#result q4
digital_prices_sample = [10, 15, 12, 18, 100, 200, 250] 
lower, upper = boot_skew(digital_prices_sample, iters=1000)
print(f"Q4 (Bootstrap Skewness) -> 95% CI: ({lower:.2f}, {upper:.2f})")

Q1 (L1 vs L2) -> Mean: 12637.5 | Median: 225.0
نکته: میانه تحت تاثیر داده پرت (50000) قرار نگرفت.

Q3 (Permutation Test) -> Obs Diff: 2.14, p-value: 0.021

Q4 (Bootstrap Skewness) -> 95% CI: (-0.56, 2.04)


<div dir="rtl" style="text-align: right;">
 
## سوال ۵ تا ۷: مفاهیم تئوری
- **سوال ۵:** مشکل آزمون چندگانه اینه که احتمال خطای نوع اول خیلی میره بالا. روش Bonferroni خیلی سختگیره و آلفا رو تقسیم بر تعداد میکنه که توان آزمون رو میاره پایین. ولی BH نرخ خطای کشف اشتباه (FDR) رو کنترل میکنه و اینطوری توان بیشتری داره.
- **سوال ۶:** تو LOOCV چون با n-1 داده آموزش میدیم مدل خیلی شبیه مدل نهاییه پس بایاس کمه. ولی چون دیتای آموزش تو هر تکرار تقریبا همونه خروجی‌ها خیلی به هم کورلیشن دارن و واریانس میره بالا.
- **سوال ۷:** معیار BIC جریمه خیلی سنگین‌تری واسه پیچیدگی مدل داره (ضریب ln(n) جای عدد ثابت). واسه همین تو دیتای بزرگ مدل‌های ساده‌تر رو ترجیح میده که اورفیت نشه.



---

## توضیح و دفاعیه بخش اول

**توضیح کدها:**
کدها رو خیلی خلاصه و منسجم و بدون هیچ پکیجی زدم. واسه پردازش دیتابیس جداول شکسته شدن تا افزونگی از بین بره. واسه اعداد رندوم تو بخش آمار هم یه تابع ژنراتور (yield) نوشتم که سبکه. توابع میانگین، میانه و چولگی رو هم با حلقه‌های ساده و لیست کامپریهنشن پایتون درآوردم.

**دلیل انتخاب راه‌حل‌ها:**
واسه اینکه به قانون ممنوعیت پکیج پایبند باشم همه‌چیو از بیس نوشتم. پردازش فایل csv با ترکیب دیکشنری‌ها انجام شد که تو رم بهینه باشه. استفاده از توزیع‌های غیرپارامتریک تو بخش آمار مثل بوت‌استرپ و جایگشت واسه این بود که دیتای فروشگاه معمولاً نرمال نیست (مخصوصاً قیمت‌ها چولگی شدیدی دارن) پس اگه با روش‌های کلاسیک می‌رفتم نتیجه کلاً اشتباه می‌شد. مقاومت میانه (L1) در برابر دیتای پرت هم کاملاً با ذات دیتای مالی همخوانی داره و بهترین انتخابه.

<div align="center">

</div>

</div>

---
SECTION 2: DigiMart
---

<div dir="rtl" style="text-align: right;">
 
## فاز ۱ و ۲: طراحی پایگاه داده و SQL
**نرمال‌سازی دیتابیس:**
- فرم اول (1NF): در جدول خام، ستون order_id به دلیل وجود چند محصول در یک سفارش تکراری است. برای یکتایی رکوردها، ترکیب order_id و product_name را به عنوان کلید اصلی (Primary Key) در نظر می‌گیریم تا اتمیسیته حفظ شود .
- فرم دوم (2NF): اطلاعات کاربران شامل نام، ایمیل و شهر دارای وابستگی جزئی به کلید اصلی هستند. برای رفع افزونگی، این اطلاعات را به جدول مستقل Users منتقل می‌کنیم .
- فرم سوم (3NF): ستون seller_city وابستگی انتقالی به seller_name دارد (نه به سفارش). پس جدول Sellers نیز جدا می‌شود تا دیتابیس کاملاً به 3NF برسد .

**کوئری‌های  SQL:**

<div align="center">

</div>

</div>

```sql
-- a.1: inner join
SELECT u.user_name, p.product_name, s.seller_name
FROM Orders o
INNER JOIN Order_Items oi ON o.order_id = oi.order_id
INNER JOIN Users u ON o.user_email = u.user_email
INNER JOIN Sellers s ON oi.seller_name = s.seller_name;

-- a.2: left join
SELECT u.user_name, COUNT(o.order_id) AS total_orders
FROM Users u
LEFT JOIN Orders o ON u.user_email = o.user_email
GROUP BY u.user_name;

-- d: upsert user 
INSERT INTO Users (user_email, user_name, user_city)
VALUES ('test@mail.com', 'ali', 'tehran')
ON CONFLICT (user_email) DO UPDATE 
SET user_name = EXCLUDED.user_name, user_city = EXCLUDED.user_city;
```


<div dir="rtl" style="text-align: right;">
 
 ## فاز ۳ و ۴: پردازش متن و پایتون


<div align="center">

</div>

</div>

In [5]:
import json
from datetime import datetime

# base product class
class product:
    def __init__(self, pid, name, price, cat):
        self.id = pid
        self.name = name
        self.cat = cat
        self.price = float(price) if price else 0.0
        
    def discount(self, pct):
        if 0 <= pct <= 100:
            self.price *= (1 - pct/100)
            
    def is_expensive(self, thresh=5000000):
        return self.price > thresh
        
    def __repr__(self):
        return f"{self.name} - {self.price}"
        
    def __lt__(self, other):
        return self.price < other.price

# digital product subclass with inheritance
class digitalproduct(product):
    def __init__(self, pid, name, price, cat, ltype, url):
        super().__init__(pid, name, price, cat)
        self.ltype = ltype
        self.url = url
        
    def generate_license(self):
        # generate 16 digit random license based on time hash
        h = abs(hash(str(datetime.now()) + self.name))
        return str(h)[:16].zfill(16)
        
    def discount(self, pct):
        # max discount allowed is 30 percent
        super().discount(min(pct, 30))

# order processor without pandas
class orderprocessor:
    def __init__(self, path):
        self.data = []
        with open(path, 'r', encoding='utf-8') as f:
            # handle bom character safely
            headers = f.readline().strip().replace('\ufeff', '').split(',')
            for line in f:
                val = line.strip().split(',')
                if len(val) == len(headers):
                    self.data.append(dict(zip(headers, val)))
                    
    def parse_date(self, dstr):
        try:
            return datetime.strptime(dstr, '%Y-%m-%d')
        except:
            return None
            
    def calc_revenue(self, month, year):
        total = 0.0
        for r in self.data:
            dt = self.parse_date(r.get('order_date', ''))
            if dt and dt.month == month and dt.year == year:
                try:
                    price = float(r.get('price', 0))
                    qty = int(r.get('quantity', 0))
                    if price > 0 and qty > 0:
                        total += price * qty
                except:
                    continue
        return total
        
    def get_delivery_durations(self):
        # calculate days between order and delivery
        durations = []
        for r in self.data:
            od = self.parse_date(r.get('order_date', ''))
            dd = self.parse_date(r.get('delivery_date', ''))
            if od and dd:
                durations.append((dd - od).days)
        return durations

# feature engineering and nlp processing 
def process_reviews(data):
    # stop words removal and bag of words without nltk/scikit
    stops = {'و', 'به', 'از', 'در', 'اب', 'این', 'با', 'رو', 'که'}
    bow = {}
    for r in data:
        text = r.get('review_text', '')
        if text:
            # simple normalization keeping only alphanumeric
            words = ''.join(c if c.isalnum() else ' ' for c in text).split()
            for w in words:
                if w not in stops:
                    bow[w] = bow.get(w, 0) + 1
                    
    top_20 = sorted(bow.items(), key=lambda x: x[1], reverse=True)[:20]
    return top_20, bow

def extract_user_features(data, user_email):
    # build user profile based on transaction history
    user_orders = [r for r in data if r.get('user_email') == user_email]
    if not user_orders:
        return None
    
    total_orders = len(user_orders)
    total_spent = sum(float(r.get('price', 0)) for r in user_orders if r.get('price'))
    
    cats = [r.get('category') for r in user_orders]
    fav_cat = max(set(cats), key=cats.count) if cats else None
    
    return {
        'total_orders': total_orders,
        'avg_order_value': total_spent / total_orders if total_orders else 0,
        'favorite_category': fav_cat
    }

In [12]:
# TEST

# testing classes
p = product("P001", "Laptop", 50000000, "Electronics")
p.discount(10)
print("Product Test (after 10% discount):", p)

dp = digitalproduct("D001", "Python Course", 1000000, "Digital", "Lifetime", "http..")
print("Generated Digital License:", dp.generate_license())
print("-" * 30)

# testing main og file
try:
    processor = orderprocessor('digimart_raw_data.csv')
    print(f"Total Rows Loaded: {len(processor.data)}")
    
    # تست درآمد ماه 5 سال 2024
    revenue = processor.calc_revenue(5, 2024)
    print(f"Revenue for 2024-05: {revenue:,.0f} Tomans")
    
    # nlp test for processing comments(5 کلمه پرتکرار اول)
    top_words, _ = process_reviews(processor.data)
    print("Top 5 Words in Reviews:", top_words[:5])
    
    # user feature engineering - تست فیچر اینجینیرینگ کاربر 
    #testing one of em emails
    sample_email = processor.data[0].get('user_email') if processor.data else None
    user_features = extract_user_features(processor.data, sample_email)
    print(f"User Features ({sample_email}):", user_features)

except FileNotFoundError:
    print("lmao go find that file lil bro.")

Product Test (after 10% discount): Laptop - 45000000.0
Generated Digital License: 1044953546544191
------------------------------
Total Rows Loaded: 5000
Revenue for 2024-05: 14,263,012,788 Tomans
Top 5 Words in Reviews: [('بود', 1147), ('کیفیت', 876), ('محصول', 552), ('خوبی', 442), ('راضی', 437)]
User Features (زهرا655hotmail): {'total_orders': 1, 'avg_order_value': 2309343.0, 'favorite_category': 'خانه و آشپزخانه'}


---
<div dir="rtl" style="text-align: right;">
 
 ## توضیح و دفاعیه بخش دوم

توضیح کوئری‌ها (بخش SQL):
کوئری‌های دیتابیس رو خیلی سرراست و بهینه زدم. جدول فلت اولیه رو تا سطح 3NF شکستم تا افزونگی اطلاعات یوزرها و فروشنده‌ها کلاً از بین بره. واسه گزارش‌گیری‌ها از JOINهای استاندارد استفاده کردم و واسه ثبت کاربر هم دستور UPSERT (همون ON CONFLICT DO UPDATE) رو پیاده کردم.

دلیل انتخاب راه‌حل‌ها:
واسه اینکه دیتابیس تو حجم بالا به مشکل نخوره و باگ‌های آپدیت و حذف نده، جداول رو کاملاً نرمال کردم؛ اینجوری مثلاً اگه شهر یه فروشنده عوض بشه، فقط همون یه رکورد آپدیت میشه نه هزار تا رکورد سفارشِ مربوط بهش. استفاده از UPSERT هم واسه این بود که تو سیستم‌های فروشگاهی خیلی وقت‌ها دیتای تکراری میاد؛ پس به جای اینکه دیتابیس ارور بده یا هندل کردن ارور رو بندازم گردن کد پایتون، همونجا تو سطح دیتابیس هندلش کردم که اگه ایمیل تکراری بود، فقط دیتاش رو آپدیت کنه. کلاً سینتکس‌ها رو استاندارد زدم که روی هر دیتابیسی راحت و سریع ران بشه.

<div align="center">

</div>

</div>